# Build annual and seasonal model inputs

## 1. Build observed monthly seasonal training tables

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from os import environ

configured_root = environ.get("WGMS_MB_ROOT")
ROOT_DIR = Path(configured_root).expanduser().resolve() if configured_root else Path.cwd().resolve()
if not configured_root:
    if (ROOT_DIR / "01_Dataset.ipynb").is_file():
        ROOT_DIR = ROOT_DIR.parent
    if not (ROOT_DIR / "code" / "01_Dataset.ipynb").is_file():
        raise RuntimeError("Start this notebook from the release root or its code directory, or set WGMS_MB_ROOT.")
elif not ROOT_DIR.is_dir():
    raise FileNotFoundError(f"The configured WGMS_MB_ROOT directory does not exist: {ROOT_DIR}")
WGMS_DIR = ROOT_DIR / "data/DOI-WGMS-FoG-2026-02-10/data"
TARGET_CSV = ROOT_DIR / "data/interim/wgms_targets/wgms_global_targets_with_rgi_1979_2025.csv"
ERA_LONG_CSV = ROOT_DIR / "data/raw/era5land_gee/wgms_global_era5land_1979_2025.csv"
OUT_DIR = ROOT_DIR / "data/processed/seasonal_training"
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_MIN, YEAR_MAX = 1979, 2025
STATE_VARS = ['t2m', 'rsn', 'sde', 'tsn', 'fal', 'sp']
FLOW_VARS = ['sf', 'smlt', 'slhf', 'ssr', 'strd', 'sshf', 'ssrd', 'tp']
CLIMATE_VARS = STATE_VARS + FLOW_VARS


def parse_wgms_date(series):
    s = series.astype('string').str.strip()
    out = pd.to_datetime(pd.Series(pd.NaT, index=series.index))
    numeric_mask = s.str.fullmatch(r'\d{8}').fillna(False)
    out.loc[numeric_mask] = pd.to_datetime(s.loc[numeric_mask], format='%Y%m%d', errors='coerce')
    out.loc[~numeric_mask] = pd.to_datetime(s.loc[~numeric_mask], errors='coerce')
    return out


def shift_month(date_series, direction):
    adjusted = date_series.copy()
    if direction == 'start':
        adjusted = adjusted.where(adjusted.dt.day <= 15, adjusted + pd.offsets.MonthBegin(1))
    elif direction == 'end':
        adjusted = adjusted.where(adjusted.dt.day >= 15, adjusted - pd.offsets.MonthEnd(1))
    else:
        raise ValueError("direction must be 'start' or 'end'")
    return adjusted


def normalize_region(series):
    return pd.to_numeric(series, errors='coerce').astype('Int64')


def choose_mode(values):
    vals = [int(v) for v in values if pd.notna(v)]
    if not vals:
        return np.nan
    vc = pd.Series(vals).value_counts()
    top_count = vc.iloc[0]
    top_vals = sorted(int(v) for v in vc[vc == top_count].index.tolist())
    if len(top_vals) == 1:
        return top_vals[0]
    return int(round(float(np.median(top_vals))))


def month_sequence(start_month, start_offset, end_month, end_offset, balance_year):
    start_year = int(balance_year) + int(start_offset)
    end_year = int(balance_year) + int(end_offset)
    start_period = pd.Period(year=start_year, month=int(start_month), freq='M')
    end_period = pd.Period(year=end_year, month=int(end_month), freq='M')
    if end_period < start_period:
        return []
    periods = pd.period_range(start_period, end_period, freq='M')
    return [(int(p.year), int(p.month)) for p in periods]


def aggregate_month_window(records, era_lookup):
    rows = []
    for rec in records.itertuples(index=False):
        months = month_sequence(rec.season_start_month, rec.season_start_year_offset, rec.season_end_month, rec.season_end_year_offset, rec.year)
        values = []
        for year_month in months:
            key = (rec.RGIId, year_month[0], year_month[1])
            if key in era_lookup:
                values.append(era_lookup[key])
        if len(values) == 0:
            out = {var: np.nan for var in CLIMATE_VARS}
        else:
            frame = pd.DataFrame(values)
            out = {var: frame[var].mean() for var in STATE_VARS}
            out.update({var: frame[var].sum(min_count=1) for var in FLOW_VARS})
        rows.append(out)
    return pd.DataFrame(rows)


def prepare_base_tables():
    mass_balance = pd.read_csv(WGMS_DIR / 'mass_balance.csv', low_memory=False)
    target_meta = pd.read_csv(TARGET_CSV, low_memory=False)
    era_long = pd.read_csv(ERA_LONG_CSV, low_memory=False)

    target_meta['wgms_glacier_id'] = pd.to_numeric(target_meta['wgms_glacier_id'], errors='coerce').astype('Int64')
    target_meta['O1Region'] = normalize_region(target_meta['O1Region'])
    target_meta['O2Region'] = pd.to_numeric(target_meta['O2Region'], errors='coerce').astype('Int64')
    mapped_ids = target_meta['wgms_glacier_id'].dropna().unique().tolist()

    base_meta_cols = [
        'RGIId', 'GLIMSId', 'CenLon', 'CenLat', 'O1Region', 'O2Region', 'Area',
        'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax', 'Status', 'Connect',
        'Form', 'TermType', 'Surging', 'Linkages', 'Name', 'gtng_region',
        'glacier_name', 'wgms_glacier_id'
    ]
    meta_map = (
        target_meta[base_meta_cols]
        .dropna(subset=['wgms_glacier_id'])
        .sort_values(['RGIId'])
        .drop_duplicates(subset=['wgms_glacier_id'], keep='first')
        .reset_index(drop=True)
    )

    mb = mass_balance.copy()
    mb['glacier_id'] = pd.to_numeric(mb['glacier_id'], errors='coerce').astype('Int64')
    mb['year'] = pd.to_numeric(mb['year'], errors='coerce').astype('Int64')
    mb = mb.loc[mb['glacier_id'].isin(mapped_ids) & mb['year'].between(YEAR_MIN, YEAR_MAX)].copy()
    mb = mb.drop(columns=[c for c in ['glacier_name', 'gtng_region'] if c in mb.columns])
    mb = mb.merge(meta_map, left_on='glacier_id', right_on='wgms_glacier_id', how='inner')

    mb['begin_dt'] = parse_wgms_date(mb['begin_date'])
    mb['mid_dt'] = parse_wgms_date(mb['midseason_date'])
    mb['end_dt'] = parse_wgms_date(mb['end_date'])
    mb['O1Region'] = normalize_region(mb['O1Region'])
    mb['O2Region'] = pd.to_numeric(mb['O2Region'], errors='coerce').astype('Int64')

    usecols = ['RGIId', 'year', 'month'] + CLIMATE_VARS
    era = era_long[usecols].copy()
    era['year'] = pd.to_numeric(era['year'], errors='coerce').astype(int)
    era['month'] = pd.to_numeric(era['month'], errors='coerce').astype(int)
    era_lookup = {
        (row.RGIId, int(row.year), int(row.month)): {var: getattr(row, var) for var in CLIMATE_VARS}
        for row in era.itertuples(index=False)
    }
    return mb, era_lookup


def build_observed_window_frame(df, season):
    work = df.copy()
    if season == 'winter':
        start = work['begin_dt']
        end = work['mid_dt']
        target_col = 'winter_balance'
    else:
        start = work['mid_dt']
        end = work['end_dt']
        target_col = 'summer_balance'

    valid = start.notna() & end.notna() & (start <= end)
    shifted_start = shift_month(start.loc[valid], direction='start')
    shifted_end = shift_month(end.loc[valid], direction='end')
    keep = shifted_start <= shifted_end
    valid_index = start.loc[valid].index[keep]

    obs = work.loc[valid_index, [
        'RGIId', 'year', 'O1Region', 'O2Region', 'Area', 'CenLon', 'CenLat',
        'Zmin', 'Zmax', 'Zmed', 'Slope', 'Aspect', 'Lmax', 'Status', 'Connect',
        'Form', 'TermType', 'Surging', 'Linkages', 'Name', 'gtng_region',
        'glacier_name', 'glacier_id', 'time_system', target_col
    ]].copy()
    obs['season'] = season
    obs['mb'] = pd.to_numeric(obs[target_col], errors='coerce')
    shifted_start = shifted_start.loc[valid_index]
    shifted_end = shifted_end.loc[valid_index]
    obs['season_start_month'] = shifted_start.dt.month.to_numpy()
    obs['season_end_month'] = shifted_end.dt.month.to_numpy()
    obs['season_start_year_offset'] = (shifted_start.dt.year - work.loc[valid_index, 'year'].astype(int)).to_numpy()
    obs['season_end_year_offset'] = (shifted_end.dt.year - work.loc[valid_index, 'year'].astype(int)).to_numpy()
    obs['season_window_source'] = 'observed_month_window'
    obs['window_confidence'] = 'high'
    return obs

In [ ]:
MAIN_WINTER_CSV = OUT_DIR / 'wgms_global_winter_era5land_1979_2025.csv'
MAIN_SUMMER_CSV = OUT_DIR / 'wgms_global_summer_era5land_1979_2025.csv'
COVERAGE_CSV = OUT_DIR / 'wgms_global_seasonal_window_coverage_summary.csv'
SUMMARY_JSON = OUT_DIR / 'wgms_global_seasonal_summary.json'

mb, era_lookup = prepare_base_tables()
print(f'mapped WGMS rows: {len(mb)}')
print(f'mapped glaciers: {mb["RGIId"].nunique()}')

In [ ]:
winter = build_observed_window_frame(mb, 'winter')
summer = build_observed_window_frame(mb, 'summer')

winter = winter.loc[winter['mb'].notna()].copy()
summer = summer.loc[summer['mb'].notna()].copy()

print({'winter_training_rows': len(winter), 'summer_training_rows': len(summer)})

In [ ]:
winter_features = aggregate_month_window(winter, era_lookup)
summer_features = aggregate_month_window(summer, era_lookup)
winter = pd.concat([winter.reset_index(drop=True), winter_features], axis=1)
summer = pd.concat([summer.reset_index(drop=True), summer_features], axis=1)

print('winter all-climate-missing rows:', int(winter[CLIMATE_VARS].isna().all(axis=1).sum()))
print('summer all-climate-missing rows:', int(summer[CLIMATE_VARS].isna().all(axis=1).sum()))

In [ ]:
winter.to_csv(MAIN_WINTER_CSV, index=False)
summer.to_csv(MAIN_SUMMER_CSV, index=False)

coverage = pd.DataFrame([
    {
        'season': 'winter',
        'coverage_category': 'observed_window_observed_target',
        'count': int(len(winter)),
    },
    {
        'season': 'summer',
        'coverage_category': 'observed_window_observed_target',
        'count': int(len(summer)),
    },
])
coverage.to_csv(COVERAGE_CSV, index=False)

summary = {
    'winter': {
        'rows': int(len(winter)),
        'glaciers': int(winter['RGIId'].nunique()),
        'window_sources': {k: int(v) for k, v in winter['season_window_source'].value_counts().to_dict().items()},
    },
    'summer': {
        'rows': int(len(summer)),
        'glaciers': int(summer['RGIId'].nunique()),
        'window_sources': {k: int(v) for k, v in summer['season_window_source'].value_counts().to_dict().items()},
    },
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))

## 2. Build annual model-ready inputs

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import dill


DATASET_PATH = ROOT_DIR / "data/interim/wgms_era5land_merged/wgms_global_rgi_era5land_1979_2025.csv"
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f"Run 01_Dataset.ipynb through section 4 first: {DATASET_PATH}")
MODEL_DIR = ROOT_DIR / "models/annual"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f'DATASET_PATH = {DATASET_PATH}')

In [ ]:
data = pd.read_csv(DATASET_PATH, low_memory=False)
print(data.shape)
display(data.head())

In [ ]:
meta_cols = [c for c in ['RGIId','year','GLIMSId','Name','wgms_area_m2','time_system','glacier_name','gtng_region','Area_source','wgms_glacier_id','n_records'] if c in data.columns]
check_cols = [c for c in data.columns if c not in meta_cols]
missing_count = data[check_cols].isnull().sum()
display(pd.DataFrame({'missing_n': missing_count, 'missing_pct': (missing_count/len(data)*100).round(2)}).query('missing_n > 0'))
data = data.dropna(subset=['mwe']).copy()
data = data.fillna(data.median(numeric_only=True))
print(data.shape)

In [ ]:
data_eda = data.drop(columns=[c for c in ['RGIId','GLIMSId','Name','wgms_area_m2','time_system','glacier_name','gtng_region','Area_source','wgms_glacier_id','n_records'] if c in data.columns])
corr_dict = {}


data_eda = pd.concat([data_eda, pd.DataFrame(corr_dict, index=data_eda.index)], axis=1)
display(data_eda.head())

In [ ]:
static_feats = [c for c in ['CenLon', 'CenLat', 'Area', 'Zmax', 'Zmed', 'Slope', 'Zmin', 'Lmax', 'Aspect'] if c in data_eda.columns]
temp_feats = [f'{m}_t2m' for m in range(1, 13) if f'{m}_t2m' in data_eda.columns]
temp_group_name = 't2m'
era_var_names = ['rsn', 'sde', 'sf', 'smlt', 'tsn', 'fal', 'slhf', 'ssr', 'strd', 'sshf', 'ssrd', 'sp', 'tp']
era_feat_groups = {var: [f'{m}_{var}' for m in range(1, 13) if f'{m}_{var}' in data_eda.columns] for var in era_var_names}
era_feats = [col for var in era_var_names for col in era_feat_groups[var]]
feature_columns = static_feats + temp_feats + era_feats
X = data_eda[feature_columns].to_numpy(dtype=float)
y = data_eda['mwe'].to_numpy(dtype=float)
print('feature_count =', len(feature_columns))
print('static_feats =', static_feats)
print('temperature_group =', temp_group_name, len(temp_feats))
print('era_feat_group_sizes =', {k: len(v) for k, v in era_feat_groups.items()})
print(X.shape, y.shape)

In [ ]:
payload = {'feature_columns': feature_columns, 'X_all': X, 'y_all': y, 'rgiid_all': data['RGIId'].astype(str).to_numpy(), 'year_all': data['year'].to_numpy(dtype=int), 'feat_groups': {'static': static_feats, temp_group_name: temp_feats, **era_feat_groups}, 'dataset_path': DATASET_PATH.relative_to(ROOT_DIR).as_posix(), 'area_min_km2': 0.0, 'area_all': data['Area'].to_numpy(dtype=float)}
if 'O1Region' in data.columns:
    payload['o1region_all'] = data['O1Region'].to_numpy(dtype=float)
if 'O2Region' in data.columns:
    payload['o2region_all'] = data['O2Region'].to_numpy(dtype=float)
with open(MODEL_DIR / 'preprocessed_data.pkl', 'wb') as fw:
    dill.dump(payload, fw)
summary = {'rows': int(len(data)), 'glaciers': int(pd.Series(payload['rgiid_all']).nunique()), 'year_min': int(pd.Series(payload['year_all']).min()), 'year_max': int(pd.Series(payload['year_all']).max()), 'feature_count': len(feature_columns), 'feature_groups': {k: len(v) for k, v in payload['feat_groups'].items()}}
(MODEL_DIR / 'preprocessed_data_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(summary)

## 3. Build seasonal model-ready inputs

In [ ]:
from pathlib import Path
import json
import dill
import numpy as np
import pandas as pd


DATA_DIR = ROOT_DIR / "data/processed/seasonal_training"
MODEL_DIR = ROOT_DIR / "models/annual_seasonal"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    'winter': DATA_DIR / 'wgms_global_winter_era5land_1979_2025.csv',
    'summer': DATA_DIR / 'wgms_global_summer_era5land_1979_2025.csv',
}
CLIMATE_VARS = ['t2m', 'rsn', 'sde', 'sf', 'smlt', 'tsn', 'fal', 'slhf', 'ssr', 'strd', 'sshf', 'ssrd', 'sp', 'tp']

In [ ]:
def preprocess_one(season_name, path):
    data = pd.read_csv(path, low_memory=False)
    data['O1Region'] = pd.to_numeric(data['O1Region'], errors='coerce').astype('Int64')
    data['O2Region'] = pd.to_numeric(data['O2Region'], errors='coerce').astype('Int64')

    static_feats = [c for c in ['CenLon', 'CenLat', 'Area', 'Zmax', 'Zmed', 'Slope', 'Zmin', 'Lmax', 'Aspect'] if c in data.columns]
    climate_feats = [c for c in CLIMATE_VARS if c in data.columns]
    feature_columns = static_feats + climate_feats

    model_df = data.dropna(subset=['mb'] + feature_columns).copy().reset_index(drop=True)
    X = model_df[feature_columns].to_numpy(dtype=float)
    y = model_df['mb'].to_numpy(dtype=float)

    payload = {
        'season': season_name,
        'dataset_path': path.relative_to(ROOT_DIR).as_posix(),
        'feature_columns': feature_columns,
        'feat_groups': {
            'static': static_feats,
            'climate': climate_feats,
        },
        'X_all': X,
        'y_all': y,
        'rgiid_all': model_df['RGIId'].to_numpy(dtype=object),
        'year_all': model_df['year'].to_numpy(dtype=int),
        'area_all': model_df['Area'].to_numpy(dtype=float),
        'o1region_all': model_df['O1Region'].astype('Int64').to_numpy(dtype=object),
        'o2region_all': model_df['O2Region'].astype('Int64').to_numpy(dtype=object),
    }

    summary = {
        'season': season_name,
        'rows': int(len(model_df)),
        'glaciers': int(model_df['RGIId'].nunique()),
        'year_min': int(model_df['year'].min()),
        'year_max': int(model_df['year'].max()),
        'feature_count': int(len(feature_columns)),
        'feature_groups': {
            'static': len(static_feats),
            'climate': len(climate_feats),
        },
        'window_sources': {k: int(v) for k, v in model_df['season_window_source'].value_counts().to_dict().items()},
    }
    return payload, summary

In [ ]:
all_summary = {}
for season_name, path in DATASETS.items():
    payload, summary = preprocess_one(season_name, path)
    dill.dump(payload, open(MODEL_DIR / f'preprocessed_{season_name}_data.pkl', 'wb'))
    (MODEL_DIR / f'preprocessed_{season_name}_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    all_summary[season_name] = summary

print(json.dumps(all_summary, indent=2))